This notebook is for processing amlr07-20221204 data

This glider had sever corrosion and a catastrophic failure, and so the data we have to work with are sbds/tbds and ebd files

NOTE: this notebook is currently a collection of random exploration scripts

In [ ]:
import logging
import os
from datetime import datetime
from pathlib import Path

import dbdreader
import esdglider.profiles as prof
import numpy as np
import xarray as xr
from esdglider.slocum import core, pipeline

from esdglider import aa, defaults, gcp, paths, plots, qartod, utils
from pyglider import slocum as pgslocum
from pyglider import utils as pgutils

_log = logging.getLogger(__name__)

### Variables for user to update
deployment_name = "amlr07-20221204"    # "amlr08-20220513"
mode = "delayed"        # "delayed" or "rt"
write_nc = True         # Write NC files?
sci_use_m_depth = False # Use m_depth for science depth?
prof_args = {}          # Named optional parameters for finding profiles
maxgap=60

### Consistent variables
home = Path.home()
mnt_path = home / "mnt-gcs"
# logs_bucket_name = defaults.BUCKET_LOGS
# logs_path = mnt_path / logs_bucket_name
# file_info, log_file_name = paths.get_file_info(Path(__file__))

In [ ]:
_log.info("Generating glider paths---------------------")
glider_paths = paths.get_path_glider(
    deployment_name = deployment_name, 
    mode = mode, 
    home_path = home,
    data_in_path="/home/user/amlr07-20221204", 
    data_out_path="/home/user/amlr07-20221204", 
)
# gcp.gcs_mount_bucket(defaults.BUCKET_DATA_IN, glider_paths["data_in_path"], ro=True)
# gcp.gcs_mount_bucket_check(defaults.BUCKET_DATA_OUT, glider_paths["data_out_path"], ro=False)

glider_paths["binarydir"] = "/home/user/amlr07-20221204/binary"

for key, path in glider_paths.items():
    print(f"{key}: {path}")

## Generate timeseries

In [ ]:
deploymentyaml = glider_paths["deploymentyaml"]
rawdir = glider_paths["rawdir"]
tsdir = glider_paths["tsdir"]
ancdir = glider_paths["ancillarydir"]


utils.makedirs_pass(rawdir)
utils.makedirs_pass(ancdir)

deployment = pgutils._get_deployment(deploymentyaml)
start_date = deployment["metadata"].get("start_date", None)
if start_date is None:
    _log.warning(
        "start_date is not specified in deployment metadata. "
        "All valid timestamps will be kept"
    )
elif isinstance(start_date, datetime):
    # _log.debug("start_date read as datetime object - convert back to str")
    start_date = (start_date.replace(tzinfo=None).isoformat() + "Z")
else:
    if utils.parse_iso8601(start_date) is None:
        # _log.error(
        #     "start_date (%s) is not in a valid ISO 8601 format in "
        #     "deployment metadata",
        #     start_date
        # )
        raise ValueError(
            "start_date is provided, but is not in a "
            "valid ISO 8601 format in deployment metadata"
        )

In [ ]:
raw_yaml_list = [
    deploymentyaml, 
    paths.get_path_yaml_slocum_vars("raw")
]
binary_search="*.[SsEe][Bcbc][Dd]"
prof_depth_var="m_depth"

outname_tsraw = core.binary_to_raw_timeseries(
    glider_paths["binarydir"],
    glider_paths["cacdir"],
    rawdir,
    raw_yaml_list, 
    search=binary_search,
    include_source=True,
    fnamesuffix=f"-{mode}-raw",
    prof_depth_var=prof_depth_var,
    prof_args={},
)

In [ ]:
print(outname_tsraw)

In [ ]:
# indir=glider_paths["binarydir"]
# cachedir=glider_paths["cacdir"]
# outdir=rawdir
# deploymentyaml=raw_yaml_list
# # search="*.[D|E]BD",
# search = binary_search
# include_source=True
# fnamesuffix=f"-{mode}-raw"
# # prof_depth_var=prof_depth_var
# prof_args={}



In [ ]:
tsraw = xr.load_dataset(glider_paths["tsrawpath"])
tsraw

In [ ]:
# Check monotonicity
print(tsraw.indexes["time"].is_monotonic_increasing)

# Check for duplicate timestamps
print(tsraw.indexes["time"].has_duplicates)

# Inspect the coordinate values/dtype
print(tsraw.time.values[:10])

In [ ]:
prof_summ_path = glider_paths["profsummpath"]
other_depth_var = "depth_ctd"

# Save profile summary, get profile index attributes
_log.info("Writing profile summary CSV to %s", prof_summ_path)
prof_summ = prof.calc_profile_summary(tsraw, prof_depth_var)
prof_summ.to_csv(prof_summ_path, index=False)
num_dives = np.count_nonzero(prof_summ.profile_direction.values == 1)
_log.info("Deployment %s performed %s dives", deployment_name, num_dives)

prof_index_attrs = tsraw["profile_index"].attrs

# Profile and depth sanity checks
_log.info("raw timeseries checks")
_log.info("Running profile checks on profile depth var, '%s'", prof_depth_var)
prof.check_profiles(prof_summ)

In [ ]:
_log.info("Running profile checks on other depth var, '%s'", other_depth_var)
prof_summ2 = prof.calc_profile_summary(tsraw, other_depth_var)
prof.check_profiles(prof_summ2)

In [ ]:
tseng = xr.load_dataset(outname_tseng)
tseng

In [ ]:
# Check monotonicity
print(tseng.indexes["time"].is_monotonic_increasing)

# Check for duplicate timestamps
print(tseng.indexes["time"].has_duplicates)

# Inspect the coordinate values/dtype
print(tseng.time.values[:10])

In [ ]:
tssci = xr.load_dataset(glider_paths["tsscipath"])
tssci

In [ ]:
# Check monotonicity
print(tssci.indexes["time"].is_monotonic_increasing)

# Check for duplicate timestamps
print(tssci.indexes["time"].has_duplicates)

# Inspect the coordinate values/dtype
print(tssci.time.values[:10])

In [ ]:
# Calculate difference between each timestamp and the previous one
time_diff = tssci.time.diff(dim="time")

# Find the timestamps that did not advance
bad_times = tssci.time[1:][time_diff <= np.timedelta64(0, "ns")]

print("Timestamps that are <= previous timestamp:")
print(bad_times.values)

In [ ]:
import pandas as pd

ds = tssci

# Boolean mask of steps that went backwards or stayed equal
bad_step_mask = (ds.time.diff(dim="time") <= np.timedelta64(0, "ns")).values

# Indices where the problem occurred
bad_indices = np.where(bad_step_mask)[0] + 1

comparison = pd.DataFrame(
    {
        "index": bad_indices,
        "previous_time": ds.time.values[bad_indices - 1],
        "non_increasing_time": ds.time.values[bad_indices],
    }
)

print(comparison)

In [ ]:
window = 200  # rows before and after
target_idx=708431   

start = max(0, target_idx - window)
end = min(len(ds.time), target_idx + window + 1)

subset_ds = ds.isel(time=slice(start, end))

# # Convert to a DataFrame so Data Wrangler can open it
# df_inspect = subset_ds.to_dataframe().reset_index()
subset_ds

In [ ]:
ds = tssci.sel(time=slice(np.datetime64("1970-01-01"), None))